# 🎯 Entrenamiento v2 — YOLOv8m (09/06/2026)

Reconstrucción del entrenamiento de la **presentación v2** para volver a obtener su `results.csv`.

| | |
|---|---|
| Modelo | YOLOv8 Medium (m), pre-entrenado en COCO |
| Dataset | Roboflow `ucu-udelar-2026-a`, versión 2 |
| Train / valid / test | 279 / 69 / 66 imágenes fuente (414 en total), 3 salidas por augmentación en train |
| Épocas / imagen | 100 / 640 px |
| Qué cambió | 100 épocas (antes 50) y rebalance de clases 70 / 15 / 15. Preprocesado stretch 640×640; augmentaciones: flip H/V, rotación ±15°, saturación ±20 %, brillo ±15 %, exposición ±10 %, blur 1 px, ruido 1 %. |
| Tiempo aprox. (T4) | ~1 h 08 min |
| Resultado esperado | mAP50 0,478 · mAP50-95 0,252 · col 0,425 · osa 0,455 · sau 0,555 (valid). Verificado en los train_args de best.pt. |

**Antes de empezar:**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.
2. Cargar la API key de Roboflow en los *Secrets* de Colab (ícono de llave 🔑 a la izquierda, nombre `ROBOFLOW_API_KEY`, con acceso al notebook). **No se escribe en ninguna celda.**
3. Ejecutar las celdas en orden. Todo se guarda en Drive (`MyDrive/yolo_entrenamiento/v2_yolov8m`).

---
## Celda 1 · Verificar GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

---
## Celda 2 · Instalar librerías
Se fija `ultralytics==8.4.63`, la versión con la que se entrenó en junio (figura en `best.pt` de v2). Con otra versión cambian los valores por defecto y los resultados.

In [ ]:
!pip install -q ultralytics==8.4.63 roboflow
import ultralytics
print('ultralytics', ultralytics.__version__)

---
## Celda 3 · Configuración
Valores tomados de la presentación. Lo que no se indica queda en el valor por defecto de ultralytics, como en el entrenamiento original (batch 16, optimizer auto, seed 0).

In [ ]:
VERSION     = 'v2'
MODELO_BASE = 'yolov8m.pt'
EPOCHS      = 100
IMGSZ       = 640
PATIENCE    = None          # None = valor por defecto de ultralytics (100), como en el original
USAR_SHARPENING = False

WORKSPACE = 'ucuestudiantesproyecto-gmail-com'
PROYECTO  = 'ucu-udelar-2026-a'
VERSION_DATASET = 2

DRIVE_DIR = '/content/drive/MyDrive/yolo_entrenamiento'
NOMBRE    = 'v2_yolov8m'          # nombre propio: no pisa las corridas de otras versiones

print(f'{VERSION}: {MODELO_BASE} · {EPOCHS} épocas · {IMGSZ} px · dataset {PROYECTO}/{VERSION_DATASET}')

---
## Celda 4 · Descargar el dataset de Roboflow
La API key se lee de los *Secrets* de Colab y, si no está, se pide por teclado.

In [ ]:
from roboflow import Roboflow
import os, getpass

try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass.getpass('API key de Roboflow: ')

rf = Roboflow(api_key=API_KEY.strip())
dataset = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION_DATASET).download('yolov8')
del API_KEY

DATASET_DIR = dataset.location
DATA_YAML   = os.path.join(DATASET_DIR, 'data.yaml')
for s in ('train', 'valid', 'test'):
    carpeta = os.path.join(DATASET_DIR, s, 'images')
    n = len(os.listdir(carpeta)) if os.path.isdir(carpeta) else 0
    print(f'{s:5s}: {n} imágenes')
print('esperado: 279 / 69 / 66 imágenes fuente (414 en total), 3 salidas por augmentación en train')

---
## Celda 5 · Montar Google Drive
La corrida se guarda directo en Drive, así no se pierde si Colab se desconecta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
SAVE_DIR = os.path.join(DRIVE_DIR, NOMBRE)
print('Resultados en:', SAVE_DIR)

---
## Celda 6 · 🚀 Entrenar
No cerrar la pestaña mientras entrena. Si Colab se desconecta, ejecutar las celdas 1 a 5 y después la celda **6b**: retoma desde la última época guardada.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODELO_BASE)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    project=DRIVE_DIR,
    name=NOMBRE,
    exist_ok=True,
    device=0,
)
SAVE_DIR = str(results.save_dir)
print('✅ Entrenamiento completado. Resultados en', SAVE_DIR)

### Celda 6b · (solo si Colab se desconectó) Retomar el entrenamiento

In [ ]:
# from ultralytics import YOLO
# model = YOLO(os.path.join(SAVE_DIR, 'weights', 'last.pt'))
# results = model.train(resume=True)

---
## Celda 7 · Resultados
Métricas de la mejor época según `results.csv` (la que guarda `best.pt`), validación de `best.pt` en **valid** y en **test**, y los gráficos.

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(os.path.join(SAVE_DIR, 'results.csv'))
df.columns = [c.strip() for c in df.columns]
fit = 0.1 * df['metrics/mAP50(B)'] + 0.9 * df['metrics/mAP50-95(B)']   # criterio de ultralytics para best.pt
mejor = df.loc[fit.idxmax()]
print(f"Épocas entrenadas: {len(df)} · mejor época: {int(mejor['epoch'])}")
for k in ('metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)'):
    print(f'  {k:22s} {mejor[k]:.3f}')

best = YOLO(os.path.join(SAVE_DIR, 'weights', 'best.pt'))
filas = []
for split in ('val', 'test'):
    r = best.val(data=DATA_YAML, split=split, imgsz=IMGSZ, plots=False, verbose=True)
    filas.append({'version': VERSION, 'split': split, 'precision': r.box.mp, 'recall': r.box.mr,
                  'mAP50': r.box.map50, 'mAP50_95': r.box.map,
                  **{f'mAP50_{best.names[i]}': r.box.ap50[j] for j, i in enumerate(r.box.ap_class_index)}})
resumen = pd.DataFrame(filas).round(4)
resumen.to_csv(os.path.join(SAVE_DIR, f'metricas_{VERSION}.csv'), index=False)
display(resumen)

for f in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    p = os.path.join(SAVE_DIR, f)
    if os.path.exists(p):
        print('──', f); display(Image(filename=p, width=900))

---
## Celda 8 · Descargar resultados
Un `.zip` con `results.csv`, `metricas_v2.csv`, `args.yaml`, `best.pt` y todos los gráficos. También queda todo en Drive.

In [ ]:
import shutil
from google.colab import files

stage = f'/content/{NOMBRE}'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(os.path.join(stage, 'weights'))
shutil.copy2(os.path.join(SAVE_DIR, 'weights', 'best.pt'), os.path.join(stage, 'weights', 'best.pt'))
for f in os.listdir(SAVE_DIR):
    if f.lower().endswith(('.csv', '.png', '.jpg', '.yaml')):
        shutil.copy2(os.path.join(SAVE_DIR, f), stage)
z = shutil.make_archive(f'/content/{NOMBRE}', 'zip', stage)
files.download(z)
files.download(os.path.join(SAVE_DIR, 'results.csv'))

---
## Celda 9 (opcional) · Probar con una imagen
Caso puntual de la presentación: `IMG_7717ecoli_T1_10^-6_19.JPG` (clase `col`, 16 colonias en Roboflow). Mismos parámetros de detección que en la presentación (los valores por defecto de ultralytics).

In [ ]:
from google.colab import files as colab_files
uploaded = colab_files.upload()
for nombre in uploaded:
    if USAR_SHARPENING:
        cv2.imwrite(nombre, sharpen(cv2.imread(nombre)))   # mismo preprocesado que en el entrenamiento
    r = best.predict(nombre, save=True, project='/content/predicciones', name=VERSION, exist_ok=True)[0]
    conteo = {best.names[int(k)]: int((r.boxes.cls == k).sum()) for k in r.boxes.cls.unique()}
    print(f'{nombre}: {len(r.boxes)} colonias detectadas', conteo)
    display(Image(filename=os.path.join(r.save_dir, os.path.basename(nombre)), width=700))